# KDE Lambda Calibration

**Intent:** Gut-check whether we can predict a movie's total review volume from observable features, so we can calibrate the per-critic KDE lambda model for movies that draw more/fewer reviews than average.

**Context:** See `brainstorm/brainstorm_critic_kde_lambda.md` §"KDE weighting and movie heterogeneity (2026-04-07)". Two concerns:
1. Each critic's KDE must be weighted by P(critic reviews this movie) — straightforward fix using historical review rates.
2. The weighted sum still reflects the "average" movie. Big releases will be underestimated, small ones overestimated. Need a way to scale lambda to the target movie.

**Approach:**
1. Test Kalshi window (bet open → close) as a predictor of review volume → **Result: weak (R²=0.058)**
2. Test early observed review count as a predictor of final count → find the time horizon where it becomes reliable

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from datetime import timedelta

# Load data
movies = pd.read_csv('../movies_index.csv')
reviews = pd.read_csv('../reviews.csv')
reviews['estimated_timestamp'] = pd.to_datetime(reviews['estimated_timestamp'], format='ISO8601', utc=True)

movies['Bet Open Date'] = pd.to_datetime(movies['Bet Open Date']).dt.tz_localize('UTC')
movies['Bet Close Date'] = pd.to_datetime(movies['Bet Close Date']).dt.tz_localize('UTC')

# Filter to movies with valid dates and bet close in the past
movies = movies.dropna(subset=['Bet Open Date', 'Bet Close Date'])
movies = movies[movies['Bet Close Date'] < pd.Timestamp('2026-04-07', tz='UTC')].copy()

# Kalshi window in days
movies['kalshi_window_days'] = (movies['Bet Close Date'] - movies['Bet Open Date']).dt.days

# Review cutoff: 24h before bet close date
movies['review_cutoff'] = movies['Bet Close Date'] - timedelta(days=1)

# Count reviews per movie up to cutoff
review_counts = []
first_review_times = []
for _, row in movies.iterrows():
    slug = row['Slug']
    cutoff = row['review_cutoff']
    movie_reviews = reviews[(reviews['movie_slug'] == slug) & (reviews['estimated_timestamp'] < cutoff)]
    review_counts.append(len(movie_reviews))
    first_review_times.append(movie_reviews['estimated_timestamp'].min() if len(movie_reviews) > 0 else pd.NaT)

movies['review_count'] = review_counts
movies['first_review'] = first_review_times
movies = movies[movies['review_count'] > 0].copy()

print(f"Movies for analysis: {len(movies)}")
print(f"Review count: mean={movies['review_count'].mean():.0f}, median={movies['review_count'].median():.0f}, range=[{movies['review_count'].min()}, {movies['review_count'].max()}]")
print(f"Kalshi window: mean={movies['kalshi_window_days'].mean():.0f}d, median={movies['kalshi_window_days'].median():.0f}d, range=[{movies['kalshi_window_days'].min()}, {movies['kalshi_window_days'].max()}]")

## Test 1: Kalshi window vs review volume

In [ ]:
r, p = stats.pearsonr(movies['kalshi_window_days'], movies['review_count'])
rho, p_rho = stats.spearmanr(movies['kalshi_window_days'], movies['review_count'])

fig, ax = plt.subplots(figsize=(10, 7))
ax.scatter(movies['kalshi_window_days'], movies['review_count'], alpha=0.5, s=30)

slope, intercept, _, _, _ = stats.linregress(movies['kalshi_window_days'], movies['review_count'])
x_line = np.linspace(movies['kalshi_window_days'].min(), movies['kalshi_window_days'].max(), 100)
ax.plot(x_line, slope * x_line + intercept, 'r-', linewidth=2, label=f'OLS: y = {slope:.1f}x + {intercept:.1f}')

ax.set_xlabel('Kalshi Window (days: bet open -> bet close date)', fontsize=12)
ax.set_ylabel('Review Count (up to 24h before close)', fontsize=12)
ax.set_title(f'Kalshi Window vs Review Volume (n={len(movies)}, r={r:.3f}, R²={r**2:.3f}, ρ={rho:.3f})', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# Annotate outliers
for _, row in movies.nlargest(3, 'review_count').iterrows():
    ax.annotate(row['Slug'].replace('_', ' '), (row['kalshi_window_days'], row['review_count']),
                fontsize=7, alpha=0.7, xytext=(5, 5), textcoords='offset points')
for _, row in movies.nlargest(3, 'kalshi_window_days').iterrows():
    if row['Slug'] not in movies.nlargest(3, 'review_count')['Slug'].values:
        ax.annotate(row['Slug'].replace('_', ' '), (row['kalshi_window_days'], row['review_count']),
                    fontsize=7, alpha=0.7, xytext=(5, 5), textcoords='offset points')
for _, row in movies.nsmallest(5, 'kalshi_window_days').iterrows():
    ax.annotate(row['Slug'].replace('_', ' '), (row['kalshi_window_days'], row['review_count']),
                fontsize=7, alpha=0.7, xytext=(5, -12), textcoords='offset points')

plt.tight_layout()
plt.show()

print(f"\nVerdict: R²={r**2:.3f} — Kalshi window explains ~{r**2*100:.0f}% of variance. Not useful as a conditioning feature.")

## Test 2: Early review count vs final count

At what point (days after first review) does the running count become a reliable predictor of the final total? This tells us how much observation time we need before we can confidently scale lambda for a specific movie.

**Note:** 97.8% of review timestamps are day-level resolution, so sub-day granularity is meaningless in this dataset. Analysis uses day-level cutoffs.

In [ ]:
days_to_check = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 14, 17, 21]

# For each movie, compute review count at each day mark after first review
for d in days_to_check:
    counts = []
    for _, row in movies.iterrows():
        slug = row['Slug']
        cutoff = row['review_cutoff']
        first = row['first_review']
        t = first + timedelta(days=d)
        # Don't count past the review cutoff
        t = min(t, cutoff)
        count = ((reviews['movie_slug'] == slug) & (reviews['estimated_timestamp'] < t)).sum()
        counts.append(count)
    movies[f'count_{d}d'] = counts

# Compute R² at each time horizon
print(f"{'Days':>6s}  {'r':>6s}  {'R²':>6s}  {'p-value':>10s}  {'Median count':>12s}  {'Median % of final':>18s}")
print("-" * 70)

r2_values = []
r_values = []
for d in days_to_check:
    col = f'count_{d}d'
    valid = movies[movies[col] > 0]
    r, p = stats.pearsonr(valid[col], valid['review_count'])
    r2 = r ** 2
    r_values.append(r)
    r2_values.append(r2)
    med_c = valid[col].median()
    med_pct = (valid[col] / valid['review_count']).median() * 100
    print(f"{d:>5d}d  {r:>6.3f}  {r2:>6.3f}  {p:>10.2e}  {med_c:>12.0f}  {med_pct:>17.0f}%")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 11))

# Panel 1: R² curve over time
ax = axes[0, 0]
ax.plot(days_to_check, r2_values, 'b-o', markersize=5)
ax.set_xlabel('Days after first review')
ax.set_ylabel('R²')
ax.set_title('How quickly does early count predict final count?')
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1)
ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='R²=0.5')
ax.legend()

# Panels 2-4: Scatter plots at representative time points
scatter_days = [2, 3, 7]
for ax, d in zip([axes[0, 1], axes[1, 0], axes[1, 1]], scatter_days):
    col = f'count_{d}d'
    valid = movies[movies[col] > 0]
    r, _ = stats.pearsonr(valid[col], valid['review_count'])

    ax.scatter(valid[col], valid['review_count'], alpha=0.5, s=25)
    slope, intercept, _, _, _ = stats.linregress(valid[col], valid['review_count'])
    x_line = np.linspace(valid[col].min(), valid[col].max(), 100)
    ax.plot(x_line, slope * x_line + intercept, 'r-', linewidth=2)
    # 1:1 reference line
    lim = max(valid[col].max(), valid['review_count'].max())
    ax.plot([0, lim], [0, lim], 'k--', alpha=0.3, label='1:1')

    ax.set_xlabel(f'Reviews at day {d}')
    ax.set_ylabel('Final review count')
    ax.set_title(f'Day {d}: r={r:.3f}, R²={r**2:.3f}')
    ax.grid(True, alpha=0.3)
    ax.legend()

plt.tight_layout()
plt.show()

## Test 3: Day-1 review rate (normalized by window) vs final count

Does normalizing day-1 count by the review window (first review → bet close) produce a better predictor? Intuition: 22 reviews on day 1 of a 5-day window is very different from 22 reviews on day 1 of a 20-day window.

In [ ]:
# Review window: first review → bet close date (in days)
movies['review_window_days'] = (movies['Bet Close Date'] - movies['first_review']).dt.total_seconds() / 86400

valid = movies[(movies['count_1d'] > 0) & (movies['review_window_days'] > 0)].copy()

# Day-1 and Day-2 rates
valid['day1_rate'] = valid['count_1d'] / valid['review_window_days']
valid['day2_rate'] = valid['count_2d'] / valid['review_window_days']

fig, axes = plt.subplots(2, 2, figsize=(14, 11))

# Top left: day-1 rate
r1, _ = stats.pearsonr(valid['day1_rate'], valid['review_count'])
rho1, _ = stats.spearmanr(valid['day1_rate'], valid['review_count'])
ax = axes[0, 0]
ax.scatter(valid['day1_rate'], valid['review_count'], alpha=0.5, s=30)
slope, intercept, _, _, _ = stats.linregress(valid['day1_rate'], valid['review_count'])
x_line = np.linspace(valid['day1_rate'].min(), valid['day1_rate'].max(), 100)
ax.plot(x_line, slope * x_line + intercept, 'r-', linewidth=2, label=f'OLS: y = {slope:.1f}x + {intercept:.1f}')
ax.set_xlabel('Day-1 Reviews / Review Window (days)', fontsize=11)
ax.set_ylabel('Final Review Count', fontsize=11)
ax.set_title(f'Day-1 Rate (r={r1:.3f}, R²={r1**2:.3f}, ρ={rho1:.3f})', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Top right: day-2 rate
valid_d2 = valid[valid['count_2d'] > 0]
r2, _ = stats.pearsonr(valid_d2['day2_rate'], valid_d2['review_count'])
rho2, _ = stats.spearmanr(valid_d2['day2_rate'], valid_d2['review_count'])
ax = axes[0, 1]
ax.scatter(valid_d2['day2_rate'], valid_d2['review_count'], alpha=0.5, s=30)
slope, intercept, _, _, _ = stats.linregress(valid_d2['day2_rate'], valid_d2['review_count'])
x_line = np.linspace(valid_d2['day2_rate'].min(), valid_d2['day2_rate'].max(), 100)
ax.plot(x_line, slope * x_line + intercept, 'r-', linewidth=2, label=f'OLS: y = {slope:.1f}x + {intercept:.1f}')
ax.set_xlabel('Day-2 Reviews / Review Window (days)', fontsize=11)
ax.set_ylabel('Final Review Count', fontsize=11)
ax.set_title(f'Day-2 Rate (r={r2:.3f}, R²={r2**2:.3f}, ρ={rho2:.3f})', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Bottom left: raw day-1 count comparison
r1_raw, _ = stats.pearsonr(valid['count_1d'], valid['review_count'])
ax = axes[1, 0]
ax.scatter(valid['count_1d'], valid['review_count'], alpha=0.5, s=30)
slope, intercept, _, _, _ = stats.linregress(valid['count_1d'], valid['review_count'])
x_line = np.linspace(valid['count_1d'].min(), valid['count_1d'].max(), 100)
ax.plot(x_line, slope * x_line + intercept, 'r-', linewidth=2, label=f'OLS: y = {slope:.1f}x + {intercept:.1f}')
ax.set_xlabel('Raw Day-1 Review Count', fontsize=11)
ax.set_ylabel('Final Review Count', fontsize=11)
ax.set_title(f'Raw Day-1 Count (R²={r1_raw**2:.3f}) [comparison]', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Bottom right: raw day-2 count comparison
r2_raw, _ = stats.pearsonr(valid_d2['count_2d'], valid_d2['review_count'])
ax = axes[1, 1]
ax.scatter(valid_d2['count_2d'], valid_d2['review_count'], alpha=0.5, s=30)
slope, intercept, _, _, _ = stats.linregress(valid_d2['count_2d'], valid_d2['review_count'])
x_line = np.linspace(valid_d2['count_2d'].min(), valid_d2['count_2d'].max(), 100)
ax.plot(x_line, slope * x_line + intercept, 'r-', linewidth=2, label=f'OLS: y = {slope:.1f}x + {intercept:.1f}')
ax.set_xlabel('Raw Day-2 Review Count', fontsize=11)
ax.set_ylabel('Final Review Count', fontsize=11)
ax.set_title(f'Raw Day-2 Count (R²={r2_raw**2:.3f}) [comparison]', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nDay-1 rate: R²={r1**2:.3f}  |  Raw day-1: R²={r1_raw**2:.3f}")
print(f"Day-2 rate: R²={r2**2:.3f}  |  Raw day-2: R²={r2_raw**2:.3f}")
print(f"Review window: mean={valid['review_window_days'].mean():.1f}d, median={valid['review_window_days'].median():.1f}d")